# Laboratorio L15 - Estensioni, codice generato, privilegi di un agente

B.4 - Cybersicurezza e IA

Tre controlli pratici per usare in sicurezza assistenti e agenti di IA. Tutti i nomi sono fittizi.

## 1. I permessi di un'estensione del browser (esercizio 2)

Molte estensioni "di IA" per il browser chiedono permessi molto ampi. Il file `manifest.json` di un'estensione elenca i permessi richiesti. La cella analizza il manifesto di un'estensione immaginaria che promette di "riassumere le pagine con l'IA".

I nomi dei permessi sono quelli reali delle estensioni di Chrome; le descrizioni sono semplificate.

In [ ]:
import json

MANIFESTO = json.loads("""{
  "name": "Riassunti IA Pro",
  "version": "1.3",
  "permissions": ["tabs", "storage", "cookies", "clipboardRead", "history", "webRequest", "scripting"],
  "host_permissions": ["<all_urls>"]
}""")

SIGNIFICATO = {
    "tabs": ("medio", "vede indirizzo e titolo di tutte le schede aperte"),
    "storage": ("basso", "salva dati propri dell'estensione"),
    "cookies": ("alto", "legge i cookie dei siti, compresi quelli di sessione"),
    "clipboardRead": ("alto", "legge ciò che viene copiato, comprese password copiate"),
    "history": ("alto", "legge la cronologia di navigazione"),
    "webRequest": ("alto", "osserva il traffico verso i siti"),
    "scripting": ("alto", "inserisce codice nelle pagine"),
    "<all_urls>": ("alto", "agisce su tutti i siti, compresi banca, posta, registro elettronico"),
}

richiesti = MANIFESTO["permissions"] + MANIFESTO["host_permissions"]
for p in richiesti:
    livello, descrizione = SIGNIFICATO.get(p, ("?", "da verificare nella documentazione"))
    print(f"{p:15} {livello:6} {descrizione}")
print()
print("permessi ad alto rischio:", sum(SIGNIFICATO.get(p, ("?",))[0] == "alto" for p in richiesti), "su", len(richiesti))

**Domande**

- Quali permessi servono davvero per riassumere la pagina che l'utente sta guardando? (Suggerimento: esiste il permesso `activeTab`, che dà accesso solo alla scheda attiva e solo quando l'utente fa clic sull'estensione.)
- Scrivere nella cella seguente il manifesto minimo e ripetere l'analisi.

Risposte:

In [ ]:
MINIMO = {"permissions": ["activeTab"], "host_permissions": []}
SIGNIFICATO["activeTab"] = ("basso", "accede solo alla scheda attiva, dopo un clic dell'utente")

for p in MINIMO["permissions"] + MINIMO["host_permissions"]:
    print(p, SIGNIFICATO[p])

## 2. Rivedere codice generato da un assistente (esercizio 3)

Un assistente ha suggerito la funzione seguente per registrare gli utenti di una piccola applicazione della scuola. Il codice funziona, ma contiene un errore di sicurezza già studiato in L3.

- `hashlib.pbkdf2_hmac`: calcola un hash lento della password con un sale, adatto a memorizzare password
- `os.urandom(16)`: 16 byte casuali, usati come sale
- `hmac.compare_digest`: confronto che non rivela informazioni attraverso il tempo di esecuzione

In [ ]:
# codice suggerito
utenti = {}

def registra(nome, password):
    utenti[nome] = password          # la password viene memorizzata così com'è

def accedi(nome, password):
    return utenti.get(nome) == password

registra("m.bianchi", "girasole-treno-lampada-ottobre")
print(utenti)

**Domanda**: che cosa ottiene chi riesce a leggere il dizionario `utenti` (per esempio da un backup rubato)?

Versione corretta, da completare e verificare:

In [ ]:
import hashlib, hmac, os

utenti_sicuri = {}

def registra_sicuro(nome, password):
    sale = os.urandom(16)
    impronta = hashlib.pbkdf2_hmac("sha256", password.encode(), sale, 200_000)
    utenti_sicuri[nome] = (sale, impronta)

def accedi_sicuro(nome, password):
    if nome not in utenti_sicuri:
        return False
    sale, impronta = utenti_sicuri[nome]
    prova = hashlib.pbkdf2_hmac("sha256", password.encode(), sale, 200_000)
    return hmac.compare_digest(prova, impronta)

registra_sicuro("m.bianchi", "girasole-treno-lampada-ottobre")
print(utenti_sicuri["m.bianchi"][1].hex()[:32], "...")
print("password giusta :", accedi_sicuro("m.bianchi", "girasole-treno-lampada-ottobre"))
print("password errata :", accedi_sicuro("m.bianchi", "girasole-treno-lampada-novembre"))

**Domande**

- Registrare due utenti con la stessa password: le impronte sono uguali? Perché?
- Perché 200 000 iterazioni? Che cosa cambia per chi prova milioni di password su un archivio rubato (L5)?

Risposte:

## 3. Privilegi di un agente (esercizio 4)

Un agente di IA diventa pericoloso quando riunisce tre capacità (la "triade letale" descritta da Simon Willison, 2025):

- accesso a dati privati
- lettura di contenuti non fidati (email ricevute, pagine web, documenti di terzi)
- possibilità di comunicare verso l'esterno (inviare email, aprire link, caricare file)

Con tutte e tre, un'istruzione nascosta in un contenuto non fidato può portare l'agente a inviare dati privati all'esterno. La cella descrive quattro agenti immaginari e verifica quali condizioni sono presenti.

In [ ]:
AGENTI = {
    "assistente posta segreteria": {"legge_posta_ricevuta", "legge_registro", "invia_email"},
    "riassunto circolari":         {"legge_circolari_interne"},
    "prenotazione aule":           {"legge_calendario", "modifica_calendario"},
    "ricerca web per compiti":     {"naviga_web", "legge_documenti_studente", "carica_file"},
}

DATI_PRIVATI = {"legge_registro", "legge_documenti_studente", "legge_calendario"}
NON_FIDATI   = {"legge_posta_ricevuta", "naviga_web"}
ESTERNO      = {"invia_email", "carica_file"}

for nome, strumenti in AGENTI.items():
    condizioni = [bool(strumenti & DATI_PRIVATI), bool(strumenti & NON_FIDATI), bool(strumenti & ESTERNO)]
    esito = "TRIADE COMPLETA" if all(condizioni) else f"{sum(condizioni)} condizioni su 3"
    print(f"{nome:30} {esito}")

**Domande**

- Per ciascun agente con la triade completa, quale strumento togliereste o sottoporreste a conferma umana? Modificare il dizionario `AGENTI` e rieseguire la cella.
- L'agente "prenotazione aule" non ha la triade completa: quali altri rischi presenta?

Risposte: